In [ ]:
from pathlib import Path
import pandas as pd

# データ読み込み
DATA_PATH = Path("../../data/raw/assistments_2009_2010/skill_builder_data.csv")

# 対象スキル（ここだけ変える）
TARGET_SKILLS = [2, 37, 47, 48, 49, 50, 70, 76, 77, 81]


In [2]:
df = pd.read_csv(DATA_PATH, encoding="latin1")

df = df[
    ["user_id", "order_id", "template_id", "skill_id"]
].dropna()

for c in ["user_id", "order_id", "template_id", "skill_id"]:
    df[c] = df[c].astype(int)

# 対象スキルのみ
df_skill = df[df["skill_id"].isin(TARGET_SKILLS)].copy()
df_skill = df_skill.sort_values(["user_id", "order_id"])


/var/folders/zg/773ptkr55z99zw26dvy19_v00000gn/T/ipykernel_43417/354061575.py:1: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH, encoding="latin1")


In [3]:
def first_half_skills(user_df):
    n = len(user_df)
    first = user_df.iloc[: n // 2]
    return set(first["skill_id"].unique())

target_set = set(TARGET_SKILLS)

user_first_skills = (
    df_skill
    .groupby("user_id")
    .apply(first_half_skills)
)

valid_users = user_first_skills[
    user_first_skills.apply(lambda s: target_set.issubset(s))
].index

n_users = len(valid_users)

print(f"Valid users (first half covers all skills): {n_users}")


Valid users (first half covers all skills): 39


/var/folders/zg/773ptkr55z99zw26dvy19_v00000gn/T/ipykernel_43417/1938312698.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(first_half_skills)


In [4]:
df_valid = df_skill[df_skill["user_id"].isin(valid_users)]

n_templates = df_valid["template_id"].nunique()

print(f"Templates used by valid users: {n_templates}")


Templates used by valid users: 82


In [5]:
print("=== SUMMARY ===")
print(f"Skills: {TARGET_SKILLS}")
print(f"Users: {n_users}")
print(f"Templates: {n_templates}")


=== SUMMARY ===
Skills: [2, 37, 47, 49, 50, 70, 76, 77, 81]
Users: 39
Templates: 82


In [ ]:
# TARGET_SKILLS をすべて解いているユーザのうち、その skill_id も解いている人数


TOP_N = 30                                # 上位何件見るか

K = len(set(TARGET_SKILLS))
target_set = set(TARGET_SKILLS)

# (user, skill) をユニーク化：ユーザがそのスキルに触れたかどうかだけを使う（共起をユーザ数で数えるため）
user_skill = df.drop_duplicates(["user_id", "skill_id"])

# 1) TARGET_SKILLS をすべて含むユーザ集合（全期間）
target_hits = user_skill[user_skill["skill_id"].isin(target_set)]
users_with_all_targets = (
    target_hits.groupby("user_id")["skill_id"].nunique()
    .loc[lambda s: s == K]
    .index
)

n_users_all_targets = len(users_with_all_targets)
print(f"Users who attempted ALL TARGET_SKILLS (anytime): {n_users_all_targets:,}")

# 2) そのユーザ群におけるスキル共起（= そのスキルにも触れているユーザ数）
cooccur = (
    user_skill[user_skill["user_id"].isin(users_with_all_targets)]
    .groupby("skill_id")["user_id"].nunique()
    .sort_values(ascending=False)
)

# TARGET_SKILLS 自身は除外
cooccur = cooccur.drop(labels=list(target_set), errors="ignore")

# 3) ランキング表（追加したとき残るユーザ数の目安も同じ）
res = cooccur.head(TOP_N).reset_index()
res.columns = ["skill_id", "n_users_cooccur"]

res["cooccur_rate"] = res["n_users_cooccur"] / n_users_all_targets  # P(skill | all targets)
res["n_users_if_added"] = res["n_users_cooccur"]                    # 追加しても残る最大ユーザ数(全期間定義)

display(res)


Users who attempted ALL TARGET_SKILLS (anytime): 267


,skill_id,n_users_cooccur,cooccur_rate,n_users_if_added
0,37,246,0.921348,246
1,48,243,0.910112,243
2,96,237,0.887640,237
3,279,207,0.775281,207
4,280,205,0.767790,205
5,61,205,0.767790,205
6,278,205,0.767790,205
7,79,205,0.767790,205
8,67,204,0.764045,204
9,74,203,0.760300,203
